# 02 HadUK-Grid access and download size

Lists the archive, works out what the download costs, and verifies access by
pulling one file.

Directory listings are public and carry file sizes. File downloads redirect to a
login page unless a CEDA access token is sent as a bearer header. Create a token
at https://services.ceda.ac.uk/account/token/. The first cell asks for it if
`CEDA_ACCESS_TOKEN` is not already set, and holds it for this kernel session
only.

In [1]:
import pandas as pd

from fdh import weather as wx
from fdh.config import load_config, require

config = load_config()
BASE = require(config.weather.base_url, "weather.base_url")
print("token set:", wx.ensure_token())

token set: True


## What the archive holds

In [2]:
wx.list_directories(BASE + "/")

['airfrost/',
 'groundfrost/',
 'hurs/',
 'psl/',
 'pv/',
 'raindays10mm/',
 'raindays1mm/',
 'rainfall/',
 'sfcWind/',
 'snowLying/',
 'summerdays/',
 'sun/',
 'tas/',
 'tasmax/',
 'tasmin/']

In [3]:
daily = sorted(name for name, has_day in wx.daily_variables(BASE).items() if has_day)
{name: wx.list_directories(f"{BASE}/{name}/day/") for name in daily}

{'rainfall': ['v20260512/'],
 'tas': ['v20260512/'],
 'tasmax': ['v20260512/'],
 'tasmin': ['v20260512/']}

## Files and size for the weather window

The window runs from `weather.window_start` to `weather.release_covers_to` in
config.yaml. The annual release covers to 31 December 2025; anything later comes
from the Met Office provisional product, which is a different file with extra
compression.

In [4]:
VARIABLES = require(config.weather.variables, "weather.variables")
VERSION = require(config.weather.version_directory, "weather.version_directory")
START = config.weather.window_start
END = config.weather.release_covers_to

files = {name: wx.daily_files(BASE, name, VERSION, START, END) for name in VARIABLES}
summary = pd.DataFrame(
    [
        {
            "variable": name,
            "files": len(items),
            "gigabytes": round(wx.total_bytes(items) / 1e9, 1),
            "first": items[0].start,
            "last": items[-1].end,
        }
        for name, items in files.items()
    ]
)
summary.loc[len(summary)] = ["total", summary.files.sum(), summary.gigabytes.sum(), None, None]
summary

,variable,files,gigabytes,first,last
0,rainfall,189,13.6,2010-04-01,2025-12-31
1,tasmax,189,13.7,2010-04-01,2025-12-31
2,tasmin,189,13.8,2010-04-01,2025-12-31
3,total,567,41.1,None,None


## Two ways to take it

Mirroring keeps every NetCDF file on disk. Streaming downloads one month,
reduces it to service area means, and discards the file, which turns tens of
gigabytes into a table of a few hundred kilobytes. 

In [5]:
print(wx.wget_command(f"{BASE}/rainfall/day/{VERSION}/", "data/raw/weather"))

wget -e robots=off --mirror --no-parent -r --header "Authorization: Bearer $CEDA_ACCESS_TOKEN" -P data/raw/weather https://dap.ceda.ac.uk/badc/ukmo-hadobs/data/insitu/MOHC/HadOBS/HadUK-Grid/v1.3.2.ceda/1km/rainfall/day/v20260512/


## Verify access on one file

In [6]:
probe = files["rainfall"][0]
probe.name, probe.url, round(probe.size / 1e6, 1)

('rainfall_hadukgrid_uk_1km_day_20100401-20100430.nc',
 'https://dap.ceda.ac.uk/badc/ukmo-hadobs/data/insitu/MOHC/HadOBS/HadUK-Grid/v1.3.2.ceda/1km/rainfall/day/v20260512/rainfall_hadukgrid_uk_1km_day_20100401-20100430.nc',
 63.4)

In [7]:
path = wx.download(probe, config.paths.resolved("raw_dir") / "weather" / "probe")
print(path, round(path.stat().st_size / 1e6, 1), "MB")

D:\Chris Masters\Final Research Project\Chris-MScfire-demand-horizon\fire-demand-horizon\data\raw\weather\probe\rainfall_hadukgrid_uk_1km_day_20100401-20100430.nc 63.4 MB


## Inspect it

Dimensions, variables and time span of the probe file.

In [8]:
from netCDF4 import Dataset, num2date

with Dataset(path) as ds:
    print({name: len(dim) for name, dim in ds.dimensions.items()})
    for name, var in ds.variables.items():
        print(f"{name}: {var.dimensions} {var.dtype}")
    t = ds.variables["time"]
    print(num2date(t[0], t.units), "to", num2date(t[-1], t.units))

{'time': 30, 'projection_y_coordinate': 1450, 'projection_x_coordinate': 900, 'bnds': 2}
rainfall: ('time', 'projection_y_coordinate', 'projection_x_coordinate') float64
transverse_mercator: () int32
time: ('time',) int64
time_bnds: ('time', 'bnds') int64
projection_y_coordinate: ('projection_y_coordinate',) float64
projection_y_coordinate_bnds: ('projection_y_coordinate', 'bnds') float64
projection_x_coordinate: ('projection_x_coordinate',) float64
projection_x_coordinate_bnds: ('projection_x_coordinate', 'bnds') float64
latitude: ('projection_y_coordinate', 'projection_x_coordinate') float64
longitude: ('projection_y_coordinate', 'projection_x_coordinate') float64
2010-04-01 12:00:00 to 2010-04-30 12:00:00
